# ElectInfo on Azure Databricks — same pipeline, no Sedona

## What this shows
How ElectInfo runs the same Spark pipeline from `engines/02` on Azure Databricks, where Apache Sedona is not installed and adding it is non-trivial. The `databricks/dataframe_bridge` pattern — Spark ↔ pandas and Spark ↔ GeoPandas transforms — is the supported workaround. This notebook also covers the supporting Databricks utilities: Asset Bundle run URLs, LakeBase (Postgres) connection plumbing, and Unity Catalog DDL generation.

## Why it matters
Azure Databricks lacks Sedona and the native C geospatial stack GeoPandas / Shapely / Fiona need. `geopandas.read_file(...)` on a Databricks cluster fails in ways that are hard to diagnose. The `dataframe_bridge` pattern keeps Spark for distribution, hands off to pandas / GeoPandas for the geospatial step on the driver, hands back to Spark for the next stage.

## Prereqs
- `pip install 'siege-utilities[databricks]'`
- A Databricks workspace for the auth / secrets / Unity Catalog cells. The URL / LakeBase / DDL cells are pure-Python builders and run anywhere.

## Next
- `engines/02_distributed_spark.ipynb` — the on-cluster Spark + Sedona version.
- `engines/01_multi_engine_dataframes.ipynb` — the pandas ↔ DuckDB abstraction that underlies both.


## 1. Asset Bundle run URL builder

`build_databricks_run_url` produces a clickable workspace URL for a given
`run_id`. Pure string building — no workspace required.

In [1]:
from siege_utilities.databricks.artifacts import build_databricks_run_url

url = build_databricks_run_url(
    host='adb-1234567890123456.7.azuredatabricks.net',
    workspace_id='1234567890123456',
    run_id='918273645',
)
print(url)


adb-1234567890123456.7.azuredatabricks.net/jobs/runs/918273645?o=1234567890123456


## 2. LakeBase (Postgres) connection builders

LakeBase exposes a Postgres endpoint. These helpers build the JDBC URL, a
`~/.pgpass` entry, and a `psql` invocation from a single `conninfo` string —
useful for notebooks, CI jobs, or manual debugging sessions.

In [2]:
from siege_utilities.databricks.lakebase import (
    build_jdbc_url, build_lakebase_psql_command,
    build_pgpass_entry, parse_conninfo,
)

conninfo = (
    'host=my-lakebase.adb.example.net port=5432 '
    'dbname=analytics user=svc_reports password=REDACTED sslmode=require'
)
fields = parse_conninfo(conninfo)
print('parsed fields:', {k: v for k, v in fields.items() if k != 'password'})
print()
print('jdbc URL   :', build_jdbc_url(
    host=fields['host'], dbname=fields['dbname'], port=int(fields['port']),
))
print('pgpass line:', build_pgpass_entry(
    host=fields['host'], port=int(fields['port']),
    dbname=fields['dbname'], user=fields['user'],
    password=fields['password'],
))
print('psql cmd   :', build_lakebase_psql_command(
    host=fields['host'], user=fields['user'], dbname=fields['dbname'],
    port=int(fields['port']), sslmode=fields.get('sslmode'),
))


parsed fields: {'host': 'my-lakebase.adb.example.net', 'port': '5432', 'dbname': 'analytics', 'user': 'svc_reports', 'sslmode': 'require'}

jdbc URL   : jdbc:postgresql://my-lakebase.adb.example.net:5432/analytics
pgpass line: my-lakebase.adb.example.net:5432:analytics:svc_reports:REDACTED
psql cmd   : psql "host=my-lakebase.adb.example.net user=svc_reports dbname=analytics port=5432 sslmode=require"


## 3. Unity Catalog DDL generation

Produces the `CREATE FOREIGN TABLE` / schema-sync statements Unity Catalog
needs to expose an external table. No workspace connection required — these
are string builders you feed to your deployment pipeline.

In [3]:
from siege_utilities.databricks.unity_catalog import (
    build_foreign_table_sql, build_schema_and_table_sync_sql,
)

ddl = build_foreign_table_sql(
    catalog='acme_prod',
    schema='stage',
    table='contributions',
    connection_name='lakebase_acme',
    source_schema='public',
    source_table='contributions',
)
print(ddl)

print('\n-- sync multiple tables --')
for stmt in build_schema_and_table_sync_sql(
    catalog='acme_prod',
    schema='stage',
    connection_name='lakebase_acme',
    source_schema='public',
    tables=['contributions', 'disbursements', 'committees'],
):
    print(stmt + ';')


CREATE FOREIGN TABLE IF NOT EXISTS `acme_prod`.`stage`.`contributions`
USING CONNECTION `lakebase_acme`
OPTIONS (table 'public.contributions');

-- sync multiple tables --
CREATE SCHEMA IF NOT EXISTS `acme_prod`.`stage`;;
CREATE FOREIGN TABLE IF NOT EXISTS `acme_prod`.`stage`.`contributions`
USING CONNECTION `lakebase_acme`
OPTIONS (table 'public.contributions');;
CREATE FOREIGN TABLE IF NOT EXISTS `acme_prod`.`stage`.`disbursements`
USING CONNECTION `lakebase_acme`
OPTIONS (table 'public.disbursements');;
CREATE FOREIGN TABLE IF NOT EXISTS `acme_prod`.`stage`.`committees`
USING CONNECTION `lakebase_acme`
OPTIONS (table 'public.committees');;


## 4. Spark ↔ GeoPandas bridge (conceptual)

These cells don't execute inline — they require a running Spark session and
GeoPandas. They're here to document the canonical pattern for Azure Databricks
geospatial work, which is the main reason `databricks/` is a sibling to
`distributed/`.

```python
from siege_utilities.databricks.dataframe_bridge import (
    spark_to_geopandas, geopandas_to_spark,
)

# Distributed read (Spark): a table of points with lat/lon columns
spark_df = spark.table('stage.contributions_with_coords')

# Bring to driver as GeoPandas for a Shapely / proj-intensive step
gdf = spark_to_geopandas(
    spark_df,
    lon_col='lon', lat_col='lat',
    crs='EPSG:4326',
)
gdf_buffered = gdf.assign(geom_buf=gdf.geometry.buffer(0.01))

# Push back to Spark for downstream joins / writes
spark_df_out = geopandas_to_spark(spark, gdf_buffered)
spark_df_out.write.mode('overwrite').saveAsTable('stage.contributions_buffered')
```

Available bridge functions (see `siege_utilities/databricks/dataframe_bridge.py`):

- `pandas_to_spark(spark, pdf)`
- `spark_to_pandas(spark_df)`
- `geopandas_to_spark(spark, gdf)`
- `spark_to_geopandas(spark_df, lon_col, lat_col, crs)`


## 5. Workspace client + secrets (conceptual)

These also require a live Databricks environment:

```python
from siege_utilities.databricks.auth import get_workspace_client
from siege_utilities.databricks.secrets import (
    ensure_secret_scope, get_runtime_secret, put_secret, runtime_secret_exists,
)

wc = get_workspace_client()                      # picks up DATABRICKS_HOST/TOKEN
ensure_secret_scope(wc, scope='siege_utilities')
put_secret(wc, scope='siege_utilities', key='ga_service_account', value='{...}')
assert runtime_secret_exists('siege_utilities', 'ga_service_account')
svc = get_runtime_secret('siege_utilities', 'ga_service_account')
```


## Related

- Source: `siege_utilities/databricks/` (artifacts, auth, dataframe_bridge, lakebase, secrets, session, unity_catalog)
- Tests: `tests/test_databricks_*.py`
- Architecture rationale: `docs/INTENT.md` divergence D4; `docs/ARCHITECTURE.md` (databricks/ vs distributed/)
